# Session 5: Corporate, Data Centre Network and Cloud Architecture

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Hamnakhan123/Cybersecurity-and-Digital-Forensics/blob/main/term-1/03-computer-networks/code/session-05-datacentre-and-cloud.ipynb)

*Cybersecurity and Digital Forensics* · Prepared by **Hamna Zahid Khan**, Junior Lecturer

Run each cell from top to bottom (Shift+Enter). Every example is self-contained and safe to run in a lab or on Colab.

## `session-05_spine_leaf.py`

In [1]:
"""Session 5 - Size a data-centre network: three-tier vs spine-leaf.

Computes link counts, hop counts between servers and the oversubscription
ratio (server-facing bandwidth : uplink bandwidth) on each leaf switch.
Run:  python session-05_spine_leaf.py
"""


def spine_leaf(spines, leaves, servers_per_leaf, server_gbps, uplink_gbps):
    links = spines * leaves                      # every leaf connects to every spine
    down = servers_per_leaf * server_gbps        # traffic the servers can push into a leaf
    up = spines * uplink_gbps                    # capacity from that leaf to the fabric
    return {
        "servers": leaves * servers_per_leaf,
        "fabric_links": links,
        "max_hops": "leaf -> spine -> leaf (always 2 switch hops)",
        "oversub": down / up,
    }


def three_tier(core, aggregation, access, servers_per_access):
    # access -> 2 aggregation, aggregation -> every core
    links = access * 2 + aggregation * core
    return {"servers": access * servers_per_access, "links": links,
            "max_hops": "access -> agg -> core -> agg -> access (up to 4 switch hops)"}


if __name__ == "__main__":
    print("Spine-leaf designs (servers at 25 Gbps, uplinks at 100 Gbps):")
    print(f"{'spines':>6} {'leaves':>6} {'servers':>8} {'links':>6} {'oversub':>8}  verdict")
    for spines, leaves, per_leaf in [(2, 8, 48), (4, 8, 48), (4, 16, 32), (6, 20, 24)]:
        r = spine_leaf(spines, leaves, per_leaf, 25, 100)
        ratio = r["oversub"]
        verdict = "non-blocking" if ratio <= 1 else "ok for most apps" if ratio <= 3 else "too oversubscribed"
        print(f"{spines:>6} {leaves:>6} {r['servers']:>8} {r['fabric_links']:>6} {ratio:>7.1f}:1  {verdict}")
    print("\nPath length:", spine_leaf(4, 8, 48, 25, 100)["max_hops"])

    t = three_tier(core=2, aggregation=4, access=16, servers_per_access=24)
    print(f"\nThree-tier: {t['servers']} servers, {t['links']} uplinks, {t['max_hops']}")
    print("Add capacity: spine-leaf -> add a spine (all leaves gain bandwidth); "
          "three-tier -> bigger core boxes")

Spine-leaf designs (servers at 25 Gbps, uplinks at 100 Gbps):
spines leaves  servers  links  oversub  verdict
     2      8      384     16     6.0:1  too oversubscribed
     4      8      384     32     3.0:1  ok for most apps
     4     16      512     64     2.0:1  ok for most apps
     6     20      480    120     1.0:1  non-blocking

Path length: leaf -> spine -> leaf (always 2 switch hops)

Three-tier: 384 servers, 40 uplinks, access -> agg -> core -> agg -> access (up to 4 switch hops)
Add capacity: spine-leaf -> add a spine (all leaves gain bandwidth); three-tier -> bigger core boxes


## `session-05_storage_cloud.py`

In [2]:
"""Session 5 - Storage and cloud maths: RAID usable capacity, SAN vs NAS vs DAS,
and a simple on-premises vs public-cloud cost comparison over 3 years.

All prices are made-up teaching numbers, not quotes from any provider.
Run:  python session-05_storage_cloud.py
"""


def raid_usable(level: str, disks: int, size_tb: float) -> tuple[float, str]:
    if level == "RAID 0":
        return disks * size_tb, "no failures tolerated"
    if level == "RAID 1":
        return size_tb, f"{disks - 1} disk(s) may fail (mirror)"
    if level == "RAID 5":
        return (disks - 1) * size_tb, "1 disk may fail (parity)"
    if level == "RAID 6":
        return (disks - 2) * size_tb, "2 disks may fail (double parity)"
    if level == "RAID 10":
        return disks / 2 * size_tb, "1 disk per mirror pair may fail"
    raise ValueError(level)


def three_year_cost(tb: float):
    # on-premises: buy hardware once + power/admin each month
    onprem = 18000 + 36 * (250 + 400)
    # cloud: pay per TB-month + a compute instance, no upfront cost
    cloud = 36 * (tb * 23 + 700)
    return onprem, cloud


if __name__ == "__main__":
    print("8 x 4 TB disks:")
    for lvl in ["RAID 0", "RAID 1", "RAID 5", "RAID 6", "RAID 10"]:
        disks = 2 if lvl == "RAID 1" else 8
        usable, tol = raid_usable(lvl, disks, 4)
        raw = disks * 4
        print(f"  {lvl:<8} disks={disks}  usable {usable:>5.1f} TB of {raw:>2} TB "
              f"({usable / raw:>4.0%})  {tol}")

    print("\nOn-prem vs cloud for a small analytics system (teaching numbers, USD):")
    for tb in (5, 20, 60):
        on, cl = three_year_cost(tb)
        cheaper = "cloud" if cl < on else "on-prem"
        print(f"  {tb:>3} TB: on-prem ${on:>7,}  cloud ${cl:>9,.0f}  -> {cheaper} is cheaper")

8 x 4 TB disks:
  RAID 0   disks=8  usable  32.0 TB of 32 TB (100%)  no failures tolerated
  RAID 1   disks=2  usable   4.0 TB of  8 TB ( 50%)  1 disk(s) may fail (mirror)
  RAID 5   disks=8  usable  28.0 TB of 32 TB ( 88%)  1 disk may fail (parity)
  RAID 6   disks=8  usable  24.0 TB of 32 TB ( 75%)  2 disks may fail (double parity)
  RAID 10  disks=8  usable  16.0 TB of 32 TB ( 50%)  1 disk per mirror pair may fail

On-prem vs cloud for a small analytics system (teaching numbers, USD):
    5 TB: on-prem $ 41,400  cloud $   29,340  -> cloud is cheaper
   20 TB: on-prem $ 41,400  cloud $   41,760  -> on-prem is cheaper
   60 TB: on-prem $ 41,400  cloud $   74,880  -> on-prem is cheaper
